# Scryfall (Magic: The Gathering) Card Analysis with Naive Bayes classifiers

In [1]:
import pandas as pd

df = pd.read_json('data/oracle-cards.jsonl', lines=True)

## Data Preparation

In [2]:
df.columns

Index(['object', 'id', 'oracle_id', 'multiverse_ids', 'mtgo_id',
       'tcgplayer_id', 'cardmarket_id', 'name', 'lang', 'released_at', 'uri',
       'scryfall_uri', 'layout', 'highres_image', 'image_status',
       'image_updated_at', 'image_uris', 'mana_cost', 'cmc', 'type_line',
       'oracle_text', 'power', 'toughness', 'colors', 'color_identity',
       'keywords', 'all_parts', 'legalities', 'games', 'reserved',
       'game_changer', 'foil', 'nonfoil', 'finishes', 'oversized', 'promo',
       'reprint', 'variation', 'set_id', 'set', 'set_name', 'set_type',
       'set_uri', 'set_search_uri', 'scryfall_set_uri', 'rulings_uri',
       'prints_search_uri', 'collector_number', 'digital', 'rarity',
       'watermark', 'flavor_text', 'card_back_id', 'artist', 'artist_ids',
       'illustration_id', 'border_color', 'frame', 'frame_effects',
       'security_stamp', 'full_art', 'textless', 'booster', 'story_spotlight',
       'edhrec_rank', 'preview', 'prices', 'related_uris', 'purchase

In [3]:
# Remove all land cards
df = df[~df['type_line'].str.contains('Land')]
# Remove vanilla cards
df = df[df['oracle_text'] != '']

df = df[['oracle_id', 'name', 'colors', 'oracle_text']]

df['colors'] = df['colors'].apply(lambda x: x if isinstance(x, list) else None)
df.dropna(subset=['colors'], inplace=True)

In [4]:
df['colors'] = df['colors'].apply(lambda x: 'M' if len(x) > 1 else x[0] if x else 'C')
# Remove all multicolor cards
df = df[df['colors'] != "M"]
df.head(50)

,oracle_id,name,colors,oracle_text
0,00037840-6089-42ec-8c5c-281f9f474504,"Nissa, Worldsoul Speaker",G,"Landfall — Whenever a land you control enters,..."
1,000492bf-7eaa-4939-a51c-4eef74e4c1d1,Mine Security,R,"Trample\nWhen this creature enters, conjure a ..."
2,0004ebd0-dfd6-4276-b4a6-de0003e94237,Static Orb,C,"As long as this artifact is untapped, players ..."
3,0006faf6-7a61-426c-9034-579f2cfcfa83,Sensory Deprivation,U,Enchant creature\nEnchanted creature gets -3/-0.
4,00078ea3-0462-4a6e-b7b1-25fea012b2b7,Kavaron Consumed,R,You may put an artifact or creature card from ...
5,0007c283-5b7a-4c00-9ca1-b455c8dff8c3,Road of Return,G,Choose one —\n• Return target permanent card f...
6,000d5588-5a4c-434e-988d-396632ade42c,Storm Crow,U,Flying (This creature can't be blocked except ...
7,000d8291-d6a8-436e-9e17-7531333686a8,Snarlfang Vermin,B,Whenever Snarlfang Vermin deals combat damage ...
8,000e5d65-96c3-498b-bd01-72b1a1991850,Walking Sponge,U,{T}: Target creature loses your choice of flyi...
9,0012bc78-e69d-4a67-a302-e5fe0dfd4407,Ravnica at War,W,Exile all multicolored permanents.


In [5]:
tag_df = pd.read_json('data/oracle-tags.jsonl', lines=True)

In [ ]:
tag_df.columns

In [6]:
tag_df = tag_df[['label', 'taggings']]
tag_dict = {}
tag_labels = []
for index, row in tag_df.iterrows():
    taggings = row['taggings']
    label = row['label']
    card = False
    if isinstance(taggings, list):
        for i in range(len(taggings)):
            if(taggings[i]['oracle_id'] not in tag_dict):
                tag_dict[taggings[i]['oracle_id']] = []
            tag_dict[taggings[i]['oracle_id']].append(label)
            card = True
    if card:
        tag_labels.append(label)

In [7]:
tag_dict

{'2445e58b-87ed-4ab2-8209-a5e1f566fba7': ['tutor-creature-giant',
  'cycle-lrw-harbinger',
  'tutor-to-top',
  'triggered ability',
  'noncreature typal',
  'virtual vanilla',
  'typal-giant'],
 'fe84933e-d98f-4cbc-b272-387a80894e31': ['tutor-battle',
  'ramp',
  'castable from exile',
  'cycle-mom-invasion-signpost',
  'triggered ability',
  'tutor-land-any',
  'refund',
  'pillage effect',
  'exile-self-dfc-transform',
  'tutor-to-hand',
  'rummage',
  'synergy-battle'],
 '4feca1da-dcc8-440c-8933-bab91782b015': ['tutor-battle',
  'commander set booster cards',
  'unique mana cost',
  'cost ignorer',
  'tutor-to-battlefield'],
 '34f3b09e-b536-444d-a9f8-c0565ee84cba': ['tutor-battle',
  'burst draw',
  'cheaper than mv',
  'pure draw',
  'tutor-to-hand',
  'punny name'],
 'ccad9b56-3c31-445f-8b78-b5bbdb3d077a': ['cycle-cmr-artifact-partner',
  'keyword anthem',
  'unique protection',
  'mana value matters',
  'pair-commander',
  'gives protection',
  'synergy-artifact',
  'protects-art

In [ ]:
tagging_row_indices = []
tagging_col_indices = []
id_list = df['oracle_id'].tolist()
for key, value in tag_dict.items():
    if key not in id_list:
        continue
    id_index = id_list.index(key)
    for label in value:
        tagging_row_indices.append(id_index)
        tagging_col_indices.append(tag_labels.index(label))

## Oracle Tag Analysis

In [ ]:
import numpy as np
from scipy.sparse import csr_matrix

data = np.ones(len(tagging_row_indices), dtype=int)
sparse_mat = csr_matrix(
    (data, (tagging_row_indices, tagging_col_indices)),
    shape=(len(df), len(tag_labels)),
    dtype=int
)

df_sparse = pd.DataFrame.sparse.from_spmatrix(sparse_mat, index=df.index, columns=tag_labels)
df_with_tags = pd.concat([df, df_sparse], axis=1)

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_recall_fscore_support

x_train, x_test, y_train_info, y_test_info = train_test_split(df_with_tags[tag_labels], df_with_tags[['colors', 'name', 'oracle_id']], test_size=0.2, random_state=42)

y_train = y_train_info['colors']
y_test = y_test_info['colors']

classifier = MultinomialNB()
classifier.fit(x_train, y_train)

y_pred = classifier.predict(x_test)
p, r, f, s = precision_recall_fscore_support(y_test, y_pred, average='weighted')
print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1-score: {f:.4f}")

f_tags = f

### Confusion Matrix

In [ ]:
import seaborn as sns
from matplotlib import pyplot as plt
from sklearn.metrics import confusion_matrix
mat = confusion_matrix(y_test, y_pred)
sns.heatmap(mat.T, square=True, annot=True, fmt='d', cbar=False,
            xticklabels=classifier.classes_, yticklabels=classifier.classes_)
plt.xlabel('true label')
plt.ylabel('predicted label')

### Most confident correct and incorrect predictions for each color
#### Least confident prediction for each color

In [ ]:
y_pred_proba = classifier.predict_proba(x_test)
y_pred_conf = np.max(y_pred_proba, axis=1)
colors = df['colors'].unique()
for color in colors:
    correct_indices = np.where((y_pred == color) & (y_test == color))[0]
    if len(correct_indices) > 0:
        most_confident_index = correct_indices[np.argmax(y_pred_conf[correct_indices])]
        print(f"Most confident correct prediction for color {color}:")
        print(f"Predicted: {y_pred[most_confident_index]}, Actual: {y_test.iloc[most_confident_index]}, Confidence: {y_pred_conf[most_confident_index]}")
        print(f"Card name: {y_test_info.iloc[most_confident_index]['name']}")
        print(f"Tags: {tag_dict[y_test_info.iloc[most_confident_index]['oracle_id']]}")
        print()

for color in colors:
    incorrect_indices = np.where((y_pred != color) & (y_test == color))[0]
    if len(incorrect_indices) > 0:
        most_confident_index = incorrect_indices[np.argmax(y_pred_conf[incorrect_indices])]
        print(f"Most confident incorrect prediction for color {color}:")
        print(f"Predicted: {y_pred[most_confident_index]}, Actual: {y_test.iloc[most_confident_index]}, Confidence: {y_pred_conf[most_confident_index]}")
        print(f"Card name: {y_test_info.iloc[most_confident_index]['name']}")
        print(f"Tags: {tag_dict[y_test_info.iloc[most_confident_index]['oracle_id']]}")
        print()
        
for color in colors:
    colored_indices = np.where(y_test == color)[0]
    if len(colored_indices) > 0:
        least_confident_index = colored_indices[np.argmin(y_pred_conf[colored_indices])]
        print(f"Least confident prediction for color {color}:")
        print(f"Predicted: {y_pred[least_confident_index]}, Actual: {y_test.iloc[least_confident_index]}, Confidence: {y_pred_conf[least_confident_index]}")
        print(f"Card name: {y_test_info.iloc[least_confident_index]['name']}")
        if(y_test_info.iloc[least_confident_index]['oracle_id'] in tag_dict):
            print(f"Tags: {tag_dict[y_test_info.iloc[least_confident_index]['oracle_id']]}")
        else:
            print("Tags: None")
        print()


### Most relevant tag for each color

In [ ]:
for i, category in enumerate(classifier.classes_):
    print(f"Category: {category}")
    top_features_indices = np.argsort(classifier.feature_log_prob_[i])[-10:]
    top_features = [tag_labels[j] for j in top_features_indices]
    print(f"Top 10 features: {top_features}")
    print()

## Oracle Text Analysis

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split

df['oracle_text'] = df['oracle_text'].fillna('').astype(str).str.replace("\n", " ")   # Fill NaN values with empty strings

x_train, x_test, y_train_info, y_test_info = train_test_split(df['oracle_text'], df[['colors', 'name', 'oracle_id']], test_size=0.2, random_state=42)
y_train = y_train_info['colors']
y_test = y_test_info['colors']

count_vect = CountVectorizer()
count_vect.fit(df['oracle_text'])

feature_names = count_vect.get_feature_names_out()
feature_names[1000:1009]

x_train_counts = count_vect.transform(x_train)
x_test_counts = count_vect.transform(x_test)

In [ ]:
from sklearn.metrics import precision_recall_fscore_support
from sklearn.naive_bayes import MultinomialNB

classifier = MultinomialNB()

classifier.fit(x_train_counts, y_train)

y_pred = classifier.predict(x_test_counts)

p, r, f, s = precision_recall_fscore_support(y_test, y_pred, average='weighted')
print(f"Precision: {p:.4f}, Recall: {r:.4f}, F1-score: {f:.4f}")

f_text = f

### Most relevant words for each color

In [ ]:
feature_names = count_vect.get_feature_names_out()

for i, category in enumerate(classifier.classes_):
    print(f"Category: {category}")
    top_features_indices = np.argsort(classifier.feature_log_prob_[i])[-10:]
    top_features = [feature_names[j] for j in top_features_indices]
    print(f"Top 10 features: {top_features}")
    print()

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams.update({
    'font.family': 'DejaVu Sans', 'font.size': 10,
    'axes.edgecolor': '#9a9994', 'xtick.color': '#52514e', 'ytick.color': '#52514e'
})

labels = ['Random guess', 'Rules text', 'Oracle tags']
values = [1/6, f_text, f_tags]
colors = ['#c3c2b7', '#eb6834', '#2a78d6']

fig, ax = plt.subplots(figsize=(5.6, 2.6))
ax.barh(labels, values, color=colors, height=0.55)

for i, v in enumerate(values):
    ax.text(v + 0.01, i, f'{v:.2f}', va='center', fontsize=9)

ax.set_xlim(0, 0.8)
ax.set_xlabel('Weighted F1 score')
for s in ['top', 'right']:
    ax.spines[s].set_visible(False)
ax.grid(axis='x', color='#e6e5e0', lw=0.6)
ax.set_axisbelow(True)

fig.tight_layout()
fig.savefig('fig_model_compare.png', dpi=200)
plt.show()